In [1]:
!pip install -U transformers==4.52.4 langchain-classic huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 75.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: langchain-core
    Found ex

In [2]:
# from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain_community.document_loaders import PyPDFLoader
# from langchain_text_splitters import CharacterTextSplitter

In [3]:
from huggingface_hub import login
login()

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = torch.float16,
    device_map = "auto"
)

def GenerateText(prompt, max_length = 512, num_return_sequences = 1):
    inputs = tokenizer(prompt, return_tensors = "pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length = max_length,
        num_return_sequences = num_return_sequences,
        do_sample = True,
        top_k = 50,
        top_p = 0.95,
        temperature = 0.7
    )
    answers = []
    for output in outputs:
        answer = tokenizer.decode(output, skip_special_tokens = True)
        answers.append(answer)

    return answers

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [5]:
from langchain_core.language_models.llms import LLM
from typing import Any

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        return GenerateText(prompt, max_length=500)
    
    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

llm = CustomHFLLM()

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

# 1) Generating Application Idea

In [7]:
idea_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Come up with a creative app idea related to {topic}. Keep it short."
)

idea_chain = LLMChain(llm = llm, prompt = idea_prompt)

/tmp/ipykernel_58/1593247193.py:6: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  idea_chain = LLMChain(llm = llm, prompt = idea_prompt)


# 2) Extract Features

In [8]:
feature_prompt = PromptTemplate(
    input_variables = ["app_idea"],
    template = "Given this app idea: {app_idea}. list its top 3 features in a bullet list."
)

feature_chain = LLMChain(llm = llm, prompt = feature_prompt)

# 3) Create Tagline

In [9]:
tagline_prompt = PromptTemplate(
    input_variables = ["app_idea", "features"],
    template = (
        "Based on the app idea: {app_idea} and these features:\n{features},\n"
        "write a catchy tagline (1 sentence)."
    )
)

tagline_chain = LLMChain(llm = llm, prompt = tagline_prompt)

In [10]:
def run_all(topic: str):
    print(f" Topic: {topic}")

    app_idea = idea_chain.run(topic)
    print(f"\n App Idea: \n{app_idea.strip()}")

    features = feature_chain.run(app_idea)
    print(f"\n Features:\n{features.strip()}")

    tagline = tagline_chain.run({
        "app_idea" : app_idea,
        "features" : features
    })
    print(f"\n tagline:\n{tagline.strip()}")

In [11]:
run_all("programming")

/tmp/ipykernel_58/4226129493.py:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  app_idea = idea_chain.run(topic)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 Topic: programming


ValidationError: 1 validation error for Generation
text
  Input should be a valid string [type=string_type, input_value=['Come up with a creative...derboard recognition."'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type